In [ ]:
# Install required packages (run once in Colab)
!pip install pypuf ecdsa numpy

In [3]:
pip install --upgrade pip setuptools wheel


In [7]:
# Install required packages (run once)
!pip install --upgrade pip setuptools wheel
!pip install pypuf numpy ecdsa

In [8]:
import secrets
import hashlib
import csv
import json

import numpy as np
from pypuf.simulation import ArbiterPUF
from pypuf.io import random_inputs
from ecdsa import SigningKey, SECP256k1

In [9]:
def bytes_to_hex(b: bytes) -> str:
    return b.hex()

def generate_puf_tag(puf_id: bytes, n_bits: int = 64) -> dict:
    # 1) Instantiate one ArbiterPUF per tag with random delays
    puf = ArbiterPUF(n=n_bits)

    # 2) Generate a single ±1 challenge vector of length n_bits,
    #    providing a random seed for the challenge generator
    seed_inputs = secrets.randbits(32)
    X = random_inputs(n=n_bits, N=1, seed=seed_inputs)   # shape (1,n_bits), entries ±1

    # 3) Evaluate PUF to get ±1, then map to {0,1}
    resp_val = puf.eval(X)[0]          # +1 or -1
    resp_bit = 1 if resp_val > 0 else 0

    # 4) Derive a 32-byte seed from (puf_id || response_bit)
    seed = hashlib.sha256(puf_id + bytes([resp_bit])).digest()

    # 5) Build an ECDSA keypair on secp256k1
    sk = SigningKey.from_string(seed, curve=SECP256k1)
    vk = sk.get_verifying_key()

    return {
        "puf_id":    bytes_to_hex(puf_id),
        "challenge": bytes_to_hex(X.tobytes()),
        "response":  resp_bit,
        "seed":      bytes_to_hex(seed),
        "priv_key":  bytes_to_hex(sk.to_string()),
        "pub_key":   bytes_to_hex(vk.to_string("compressed")),
    }

def main():
    NUM_TAGS = 100
    tags = []

    for _ in range(NUM_TAGS):
        # 16-byte random PUF identifier
        puf_id = secrets.token_bytes(16)
        tags.append(generate_puf_tag(puf_id))

    # Write out CSV
    csv_path = "/content/puf_tags.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=tags[0].keys())
        writer.writeheader()
        writer.writerows(tags)
    print(f"Wrote {NUM_TAGS} tags to {csv_path}")

    # Write out JSON
    json_path = "/content/puf_tags.json"
    with open(json_path, "w") as f:
        json.dump(tags, f, indent=2)
    print(f"Wrote {NUM_TAGS} tags to {json_path}")

if __name__ == "__main__":
    main()


Wrote 100 tags to /content/puf_tags.csv
Wrote 100 tags to /content/puf_tags.json
